Training a MLP with Raw EEG/Audio Signals
===

# EEG-Audio Matching with Raw Signal MLP

## Project Overview

This notebook implements Multi-Layer Perceptron models for matching EEG brain signals with corresponding audio signals. The key characteristic of this approach is using **raw time-domain signals** without any frequency transformation.

**Research Question:** Can we match brain activity (EEG) with audio stimuli using raw signals alone, or do we need feature engineering like Gabor transforms?

**Task:** Binary classification - given an EEG clip and an audio clip, determine if they match (1) or don't match (0).

**Dataset:**
- EEG: 320 timesteps × 64 channels (brain activity recordings)
- Audio: 320 timesteps × 5 candidate audio signals
- Each EEG clip has 5 possible audio matches, only 1 is correct
- Total: 1,160 EEG clips × 5 audio = 5,800 training samples

**Raw Signal Approach:**
- **No preprocessing:** Uses time-domain signals as-is
- **Baseline model:** Helps evaluate whether Gabor/wavelet transforms add value
- **Challenge:** Raw signals have 66,560 features (320×64 EEG + 320×5 audio) - very high dimensional!
- **Solution:** Large hidden layer (512 units) to capture temporal patterns

**Comparison to Other Approaches:**
- Raw signals (this notebook): Direct time-domain processing
- Gabor transform: Time-frequency analysis with 2D convolutions
- Encoder methods: Learn embeddings with contrastive learning

In [ ]:
import torch
from torch.utils.data import DataLoader
from torch.nn.functional import relu, sigmoid, tanh
import matplotlib.pyplot as plt
from tqdm import tqdm
from clips_uniform import ClipsUniformDataset

## Data Loading and Preprocessing

The dataset uses a 5x expansion strategy: each EEG clip is paired with 5 audio candidates, creating 5 samples per EEG.

**Dataset Structure:**
- Original: 1,160 EEG clips
- Expanded: 5,800 samples (1,160 × 5)
- For each EEG clip, there are 5 EEG-audio pairs (only 1 is a true match)

**Example:**
- EEG clip #0 creates samples:
  - (EEG #0, Audio #0) → label = 1 if audio #0 is correct, else 0
  - (EEG #0, Audio #1) → label = 1 if audio #1 is correct, else 0
  - ... (5 total pairs)

This expansion strategy allows the model to learn by comparing an EEG with multiple audio candidates simultaneously.

## Loading Data

In [ ]:
# Loading data handled by PyTorch dataloader, implemented in 'clips_uniform.py'
training_data = ClipsUniformDataset(mat_file='../clips_uniform.mat', train=True)
testing_data = ClipsUniformDataset(mat_file='../clips_uniform.mat', train=False)

train_dataloader = DataLoader(training_data, batch_size=4, shuffle=True)
test_dataloader = DataLoader(testing_data, batch_size=4, shuffle=True)

### Visualization

In [ ]:
for X, y in train_dataloader:
    print(f"Shape of batch X:\t{len(X)}")

    eeg = X[0]
    audio = X[1]
    print(f"Shape of EEG:\t{eeg.shape}")
    print(f"Shape of Audio:\t{audio.shape}")

    # torch.t - Transpose
    # Plot EEG channel 9 from the first set of features of the batch
    plt.plot(torch.t(eeg[0])[:,9])
    break


## Neural Network Architecture

It is now time to implement a multilayer perceptron (MLP) with the following architecture:

$$
\begin{align*}
  \mathbf H &= \operatorname{ReLU}(\mathbf X \mathbf W^{(1)} + \mathbf b^{(1)}) \\
  \mathbf O &= \operatorname{softmax}(\mathbf H \mathbf W^{(2)} + \mathbf b^{(2)}).
\end{align*}
$$

The matrix $\mathbf X \in \mathbb R^{M \times D}$ above corresponds to a minibatch of $M$ datapoints $\mathbf x_i \in \mathbb R^D$. The matrix $\mathbf H \in \mathbb R^{M \times J}$ is the hidden layer. For this problem, we choose the width of the hidden layer to be $J = 256$. The $i$-th row of the output matrix $\mathbf O \in \mathbb R^{M \times K}$ corresponds to the prediction $\hat{\mathbf y} \in \mathbb R^K$ of the $i$-th datapoint. We one hot encode the labels and use a softmax on the last layer so that our MLP models a proper distribution $p(y_n|\mathbf x_n)$. Given an image $\mathbf x^*$, if the $k$-th entry of our output $\hat{\mathbf y}^*$ is the largest, then we predict class $k$. This corresponds to picking the class $y^*$ that maximizes $p(y^* | \mathbf x)$.

![](mlp.svg)

The entirety of implementing this architecture is just implementing the equations above. In order to do this, you will need to
1. **TODO:** initialize your parameters $\mathbf W^{(1)}, \mathbf b^{(1)}, \mathbf W^{(2)}, \mathbf b^{(2)}$
2. **TODO:** implement your activation functions $\operatorname{ReLU}$, $\operatorname{softmax}$
3. **TODO:** implement your forward pass $f(\mathbf X) = \ldots$

Machine learning models can be sensitive to weight initialization. For this architecture, initialize the weights as follows:

$$
\begin{align*}
  \mathbf {W^{(\ell)}}_{i,k} &\sim \mathcal 0.01 \, N(0, 1), \\
  \mathbf {b^{(\ell)}}_{i}\;\,\, &= 0\,.
\end{align*}
$$

In [ ]:
"""
We are given signals of dimension 320 by 69 (64 EEG plus 5 Audio), and we want to classify them as one of 5 classes.
At each step, we will be applying a linear transformation (W's) followed by an additive bias term (b's).
We then apply a non-linearity (relu or softmax).
"""

n_inputs = 66560 #D
n_hiddens = 512 #J
n_outputs = 1 #K

W1 = torch.nn.Parameter(torch.randn(size = (n_inputs, n_hiddens))) # matrix applying linear transformation
b1 = torch.nn.Parameter(torch.zeros(size = (1, n_hiddens))) # additive bias term
W2 = torch.nn.Parameter(torch.randn(size = (n_hiddens, n_outputs))) # matrix applying linear transformation
b2 = torch.nn.Parameter(torch.zeros(size = (1, n_outputs))) # additive bias term

params = [W1, b1, W2, b2]

In [ ]:
def net(X):
    """
    Implements the forward pass of our neural network. 
    
    :param X: a batch of signals of shape (batch_size, 320, 69).
    :return: a 2D numpy array of shape (batch_size, 5) where each row is a probability vector of length 5.
    """
    # Transpose and flatten EEG signals
    eeg = torch.transpose(X[0], dim0=1, dim1=2)
    eeg = torch.flatten(eeg, start_dim=1)

    # Transpose and flatten audio signals
    audio = torch.transpose(X[1], dim0=1, dim1=2)
    audio = torch.flatten(X[1], start_dim=1)

    # Concatenate signals
    X = torch.cat((eeg, audio), dim=1)
    # Parameters are float32
    X = X.to(torch.float32)
    H = tanh(X @ W1 + b1)
    O = sigmoid(H @ W2 + b2)
    return O

## Training Configuration

**Hyperparameters:**
- **Learning rate (lr):** 10 - Very high! Raw signals may need aggressive learning
- **Batch size:** 120 - Moderate batch size for stable gradients
- **Epochs:** 100 - Long training to learn complex temporal patterns
- **Hidden units:** 512 - Large capacity needed for 66,560 input features

**Loss Function:** Binary cross-entropy
- Measures how well predictions match true labels
- Formula: `-[y * log(p) + (1-y) * log(1-p)]`

**Optimizer:** Stochastic Gradient Descent (SGD)
- Updates: `param = param - lr * gradient`
- Implemented manually for educational purposes

**Why Manual Implementation?**
This notebook implements neural network training from scratch (without PyTorch's `nn.Module`) to demonstrate:
1. How forward propagation works (matrix multiplications + activations)
2. How backpropagation computes gradients automatically
3. How gradient descent updates parameters

This is educational - production code would use PyTorch's built-in `nn.Module` and optimizers.

In [ ]:
# ## Testing neural net
# for X, _ in train_dataloader:
#     print(f"{X[0].shape}")
#     print(f"{net(X).shape}")
#     break

## Training the Neural Network

We also need to be able to evaluate our model prediction $\hat {\mathbf y}$ with the true label $\mathbf y$. We can do this through with cross entropy loss. If $K$ is the number of categories we have, then $\hat {\mathbf y}$ and $\mathbf y$ are $K$-dimensional one hot vectors. With this representation, cross entropy loss is defined as $$\ell(\hat {\mathbf y}, \mathbf y) = -\sum_{k=1}^K \mathbf y_i \log \hat {\mathbf y}_i.$$ Note that the way we structured our neural network, only $\hat {\mathbf y}$ is one hot encoded, and the labels $y$ coming from the MNIST training set are integers. This means we can much more simply write $$\ell(\hat {\mathbf y}, y) = -\log \hat {\mathbf y}_y.$$

For any general loss function, we can implement the gradient descent update step: $$w \leftarrow w - \eta \, \nabla_w \mathcal L$$ for every model parameter $w$ where $\mathcal L$ is the average loss over the minibatch.

In [ ]:
def sgd(params, lr=0.1):
    """
    Implements stochastic gradient descent. For each param, subtract the gradient times the learning rate.
    The gradient for each param is stored in param.grad. After updating each param, set its gradient to zero.
    
    :param params: a list of parameters to update.
    :param lr: the learning rate.
    
    :return: None
    """
    with torch.no_grad():
      for param in params:
        #print(param.grad)
        param -= lr*param.grad
        param.grad.zero_()

In [ ]:
epochs = 100
batch_size = 120
lr = 10
train_iter = DataLoader(training_data, batch_size=12, shuffle=True)
test_iter = DataLoader(testing_data, batch_size=12, shuffle=True)


def accuracy(y_hat, y):
  with torch.no_grad():
    y_labels = y_hat.argmax(axis=1)
    correct = y_labels == y
    return correct.sum() / correct.numel()


def train(net, params, train_iter, test_iter, loss, updater):
  train_losses, train_accs = [], []
  test_losses, test_accs = [], []
  
  for epoch in tqdm(range(epochs)):
    train_loss, train_acc = 0.0, 0.0
    trials = 0

    for X, y in train_iter:
      trials += 1
      y_hat = net(X)
      l = loss(y_hat, y).mean()
      acc = accuracy(y_hat, y)

      l.backward()
      updater(params, lr)

      train_loss += l
      train_acc += acc

    train_losses.append(train_loss.item() / trials)
    train_accs.append(train_acc.item() / trials)

    test_loss, test_acc = 0.0, 0.0
    trials = 0

    y_pred = []
    y_true = []

    for X, y in test_iter:
      trials += 1
      with torch.no_grad():
        y_hat = net(X)

        # Recording for confusion matrix
        if epoch == epochs-1:
          y_pred += y_hat.argmax(axis=1)
          y_true += y
      
        l = loss(y_hat, y).mean()
        acc = accuracy(y_hat, y)

        test_loss += l
        test_acc += acc
    
    test_losses.append(test_loss.item() / trials)
    test_accs.append(test_acc.item() / trials)
  
  return train_losses, train_accs, test_losses, test_accs, y_pred, y_true

In [ ]:
"""
We are given signals of dimension 320 by 69 (64 EEG plus 5 Audio), and we want to classify them as one of 5 classes.
At each step, we will be applying a linear transformation (W's) followed by an additive bias term (b's).
We then apply a non-linearity (relu or softmax).
"""

n_inputs = 22080 #D
n_hiddens = 512 #J
n_outputs = 5 #K

W1 = torch.nn.Parameter(torch.randn(size = (n_inputs, n_hiddens))) # matrix applying linear transformation
b1 = torch.nn.Parameter(torch.zeros(size = (1, n_hiddens))) # additive bias term
W2 = torch.nn.Parameter(torch.randn(size = (n_hiddens, n_outputs))) # matrix applying linear transformation
b2 = torch.nn.Parameter(torch.zeros(size = (1, n_outputs))) # additive bias term

params = [W1, b1, W2, b2]

train_losses, train_accs, test_losses, test_accs, y_pred, y_true = train(net, params, train_iter, test_iter, loss=torch.nn.CrossEntropyLoss(), updater=sgd)

In [ ]:
plt.plot(train_losses, c='crimson', label='train loss')
plt.plot(train_accs, '--', c='crimson', label='train acc')
plt.plot(test_losses, c='navy', label='test loss')
plt.plot(test_accs, '--', c='navy', label='test acc')

plt.xlabel('epoch')
plt.grid()
plt.ylim(0, 2)
plt.xlim(0, epochs-1)
plt.legend()
plt.savefig('final_plot.pdf')
plt.show()

print(train_losses)
print(max(test_accs))